# Motor Overheating Prediction — Pipeline Modular

**Objetivo:** detectar fallos de temperatura en 6 motores servo usando datos de posición, temperatura y voltaje a 10 Hz.

**Métrica:** F1-score (clase 1) promediada por motor.

---

### Cómo usar este notebook

Ejecutar **de arriba a abajo la primera vez**. Después:
- Puedes re-ejecutar solo la celda de un motor específico (§ 7–12) sin recargar datos.
- Para cambiar features, edita `MOTOR_FEATURES` (§ 4) y re-ejecuta desde § 5.
- Para cambiar los grids de hiperparámetros, edita `GRID_*` en § 4.
- Para añadir nuevos grupos de datos, basta con copiarlos en `additional_data/` y re-ejecutar § 5b.

### Lanzamiento desde VSCode
El kernel debe ser el del `.venv` del proyecto (`TDS INDUSTRY/.venv/Scripts/python.exe`).  
Si aparece el error `ProactorEventLoop`, reinicia VSCode y elige el kernel de nuevo.

---

| Sección | Contenido |
|---------|----------|
| § 1 | Verificación de entorno y rutas |
| § 2 | Imports |
| § 3 | Funciones auxiliares (`inject_failure`, `pre_processing`) |
| § 4 | Configuración central (features, rutas, grids) |
| § 5a | Carga de datos base (training + test) |
| § 5b | Carga de datos adicionales (auto-detección de grupos) |
| § 6 | Función `train_motor()` — prueba LR, XGBoost e IsolationForest |
| § 7–12 | Entrenamiento motor a motor (celdas independientes) |
| § 13 | Resumen global de F1 y algoritmo ganador por motor |
| § 14 | Generación del CSV de submission |

## § 1 · Verificación de entorno

Comprueba que el kernel es correcto y que todas las rutas de datos existen antes de continuar.

In [1]:
import sys, os

print(f'Python : {sys.version}')
print(f'Kernel : {sys.executable}')
print()

NOTEBOOK_ROOT   = os.getcwd()
TRAIN_PATH      = os.path.join(NOTEBOOK_ROOT, 'data/training_data') + os.sep
TEST_PATH       = os.path.join(NOTEBOOK_ROOT, 'data/testing_data')  + os.sep
SAMPLE_SUB_PATH = os.path.join(NOTEBOOK_ROOT, 'sample_submission.csv')
SUBMISSION_DIR  = os.path.join(NOTEBOOK_ROOT, 'submissions')
os.makedirs(SUBMISSION_DIR, exist_ok=True)
ADDITIONAL_BASE = os.path.join(NOTEBOOK_ROOT, 'data/additional_data')   # carpeta raiz de datos extra
UTILITY_PATH    = os.path.join(NOTEBOOK_ROOT, 'kaggle_data_challenge')

checks = {
    'utility.py'                 : os.path.join(UTILITY_PATH, 'utility.py'),
    'training_data/'             : TRAIN_PATH,
    'testing_data/'              : TEST_PATH,
    'sample_submission'          : SAMPLE_SUB_PATH,
    'additional_data/ (opcional)': ADDITIONAL_BASE,
}
all_ok = True
for label, path in checks.items():
    exists = os.path.exists(path)
    mark   = 'OK   ' if exists else 'FALTA'
    print(f'  {mark}  {label:35s}  {path}')
    if not exists and 'opcional' not in label:
        all_ok = False

# Mostrar grupos de datos adicionales disponibles
if os.path.exists(ADDITIONAL_BASE):
    grupos = [d for d in os.listdir(ADDITIONAL_BASE)
              if os.path.isdir(os.path.join(ADDITIONAL_BASE, d))]
    print(f'\n  Grupos en additional_data/ ({len(grupos)} encontrados):')
    for g in grupos:
        gp      = os.path.join(ADDITIONAL_BASE, g)
        subdirs = [d for d in os.listdir(gp) if os.path.isdir(os.path.join(gp, d))]
        has_xl  = os.path.exists(os.path.join(gp, 'Test conditions.xlsx'))
        has_cp  = os.path.exists(os.path.join(gp, 'Test conditions copy.xlsx'))
        xl_mark = 'xlsx OK' if has_xl else ('xlsx (copia)' if has_cp else 'SIN xlsx')
        print(f'    {g}  —  {len(subdirs)} secuencias  —  {xl_mark}')

print()
print('Entorno OK' if all_ok else 'ATENCION: faltan rutas criticas')

Python : 3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]
Kernel : c:\Users\oscar\Desktop\TDS INDUSTRY\.venv\Scripts\python.exe

  OK     utility.py                           c:\Users\oscar\Desktop\TDS INDUSTRY\kaggle_data_challenge\utility.py
  OK     training_data/                       c:\Users\oscar\Desktop\TDS INDUSTRY\training_data\
  OK     testing_data/                        c:\Users\oscar\Desktop\TDS INDUSTRY\testing_data\
  OK     sample_submission                    c:\Users\oscar\Desktop\TDS INDUSTRY\sample_submission.csv
  OK     additional_data/ (opcional)          c:\Users\oscar\Desktop\TDS INDUSTRY\additional_data

  Grupos en additional_data/ (6 encontrados):
    additional_data_20240524_group_6  —  14 secuencias  —  xlsx OK
    additional_training_data_group_1  —  13 secuencias  —  xlsx OK
    additional_training_data_group_7  —  10 secuencias  —  xlsx OK
    _tmp_additional_data_20240524_group_6  —  13 secuencias  —  xlsx OK
    _tmp_

## § 2 · Imports

In [2]:
import warnings, itertools, shutil
import pandas as pd
import numpy  as np
import xgboost as xgb
from sklearn.ensemble        import IsolationForest
from sklearn.linear_model    import LogisticRegression
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import classification_report, f1_score
from sklearn.model_selection import train_test_split
from scipy.ndimage           import binary_closing

warnings.filterwarnings('ignore')
sys.path.insert(1, UTILITY_PATH)
from utility import read_all_test_data_from_path

print('Imports OK')
print(f'XGBoost {xgb.__version__}')

Imports OK
XGBoost 3.2.0


## § 3 · Funciones auxiliares

### `inject_failure`
Sintetiza una curva de temperatura realista en la región de fallo: sube en el primer tercio, baja en los dos tercios restantes.  
Reimplementa el algoritmo MATLAB descrito en `kaggle.md`.  
**Solo se aplica al set de entrenamiento**, nunca al de validación ni al de test.

### `pre_processing`
Limpia outliers, normaliza respecto al primer punto de la secuencia y genera features temporales:  
diferencias a 20 pasos y ventanas rodantes (media, máximo, std) de 5 y 20 pasos.

In [3]:
def inject_failure(temperature_values, label_values):
    failure_mask = (label_values == 1)
    if failure_mask.sum() == 0:
        return temperature_values.copy()
    tmp_temp   = temperature_values[failure_mask]
    n_seq      = len(tmp_temp)
    n_rise     = n_seq // 3
    n_fall     = n_seq - n_rise
    temp_start = tmp_temp[0]
    temp_end   = tmp_temp[-1]
    temp_high  = max(temp_start, temp_end) + np.random.randint(2, 11)
    rise       = np.linspace(temp_start, temp_high, n_rise + 1)[:-1]
    fall       = np.linspace(temp_high,  temp_end,  n_fall)
    result     = temperature_values.copy()
    result[failure_mask] = np.concatenate([rise, fall])
    return result


def pre_processing(df: pd.DataFrame):
    if len(df) == 0:
        return
    df['temperature'] = df['temperature'].where(df['temperature'].between(0,    100),  np.nan).ffill()
    df['voltage']     = df['voltage'].where    (df['voltage'].between    (6000, 9000), np.nan).ffill()
    df['position']    = df['position'].where   (df['position'].between   (0,    1000), np.nan).ffill()
    for col in ['temperature', 'voltage', 'position']:
        df[col] -= df[col].iloc[0]
    N_DIFF = 20
    df['temperature_diff'] = df['temperature'].diff(N_DIFF)
    df['voltage_diff']     = df['voltage'].diff(N_DIFF)
    df['position_diff']    = df['position'].diff(N_DIFF)
    for win, suffix in [(5, '5'), (20, '20')]:
        t = df['temperature']
        df[f'temperature_roll_mean_{suffix}'] = t.rolling(win, min_periods=1).mean()
        df[f'temperature_roll_max_{suffix}']  = t.rolling(win, min_periods=1).max()
        df[f'temperature_roll_std_{suffix}']  = t.rolling(win, min_periods=1).std().fillna(0)
    n = len(df)
    df['time_pct'] = np.linspace(0.0, 1.0, n) if n > 1 else np.zeros(n)
    df.fillna(0, inplace=True)


print('Funciones definidas')

Funciones definidas


## § 4 · Configuración central

- **`MOTOR_FEATURES`** — qué columnas usa cada motor.  
- **`GRID_LR` / `GRID_XGB`** — espacio de hiperparámetros del grid search supervisado.  
- **`THRESHOLDS` / `CLOSING_OPTS`** — umbrales de decisión y cierre morfológico.  
- **`ISO_PERCENTILES`** — percentiles de puntuación anomalía para IsolationForest.

`train_motor()` prueba **los tres algoritmos** en cada motor y elige el de mayor F1 de validación.

In [4]:
ROLLING_FEATS = [
    'temperature_roll_mean_5',  'temperature_roll_max_5',  'temperature_roll_std_5',
    'temperature_roll_mean_20', 'temperature_roll_max_20', 'temperature_roll_std_20',
]
MOTOR_FEATURES = {
    1: ['temperature', 'position', 'voltage', 'temperature_diff', 'time_pct'] + ROLLING_FEATS,
    2: ['temperature', 'position', 'voltage', 'time_pct'] + ROLLING_FEATS,
    3: ['position', 'temperature_diff', 'voltage', 'temperature', 'time_pct'] + ROLLING_FEATS,
    4: ['position', 'temperature', 'voltage', 'time_pct'] + ROLLING_FEATS,
    5: ['position', 'voltage', 'temperature', 'temperature_diff', 'time_pct'] + ROLLING_FEATS,
    6: ['position', 'temperature', 'temperature_diff', 'position_diff', 'time_pct'] + ROLLING_FEATS,
}

GRID_LR  = {'C': [0.01, 0.1, 1.0, 10.0]}
GRID_XGB = {
    'learning_rate'   : [0.02, 0.05],
    'n_estimators'    : [100, 150],
    'max_depth'       : [3, 4, 5],
    'min_child_weight': [1, 10],
}
THRESHOLDS      = [0.005, 0.01, 0.02, 0.03, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
CLOSING_OPTS    = [False, True]
ISO_PERCENTILES = [50, 60, 70, 75, 80, 85, 90, 92, 95, 97, 99]

models           = {}
val_f1_per_motor = {}

ks, vs = zip(*GRID_XGB.items())
n_xgb  = len(list(itertools.product(*vs)))
print('Configuracion cargada')
print(f'  LR configs por motor        : {len(GRID_LR["C"])} x {len(THRESHOLDS)} x {len(CLOSING_OPTS)} = {len(GRID_LR["C"]) * len(THRESHOLDS) * len(CLOSING_OPTS)}')
print(f'  XGB configs por motor       : {n_xgb} x {len(THRESHOLDS)} x {len(CLOSING_OPTS)} = {n_xgb * len(THRESHOLDS) * len(CLOSING_OPTS)}')
print(f'  IsoForest configs por motor : {len(ISO_PERCENTILES)} x {len(CLOSING_OPTS)} = {len(ISO_PERCENTILES) * len(CLOSING_OPTS)}')

Configuracion cargada
  LR configs por motor        : 4 x 14 x 2 = 112
  XGB configs por motor       : 24 x 14 x 2 = 672
  IsoForest configs por motor : 11 x 2 = 22


## § 5a · Carga de datos base

Carga el set de entrenamiento original y el set de test.  
Los datos adicionales se cargan por separado en § 5b.

In [5]:
print('=== Training data (base) ===')
train_df = read_all_test_data_from_path(TRAIN_PATH, pre_processing, is_plot=False)
print(f'Shape: {train_df.shape}')
print(f'Secuencias: {train_df["test_condition"].nunique()}')

print()
print('=== Test data ===')
test_df = read_all_test_data_from_path(TEST_PATH, pre_processing, is_plot=False)
print(f'Shape: {test_df.shape}')
print(f'Secuencias: {test_df["test_condition"].nunique()}')

print()
print('=== Distribucion de etiquetas (solo datos base) ===')
print(f'{"Motor":<8} {"Normal":>8} {"Fallo":>8} {"% fallo":>9}')
print('-' * 38)
for mid in range(1, 7):
    col = f'data_motor_{mid}_label'
    if col in train_df.columns:
        c0  = (train_df[col] == 0).sum()
        c1  = (train_df[col] == 1).sum()
        pct = 100 * c1 / (c0 + c1) if (c0 + c1) > 0 else 0
        flag = '  <- muy desbalanceado' if pct < 1.0 else ''
        print(f'{mid:<8} {c0:>8} {c1:>8} {pct:>8.1f}%{flag}')

=== Training data (base) ===
Shape: (39309, 86)
Secuencias: 23

=== Test data ===
Shape: (14157, 86)
Secuencias: 8

=== Distribucion de etiquetas (solo datos base) ===
Motor      Normal    Fallo   % fallo
--------------------------------------
1           37960     1349      3.4%
2           32577     6732     17.1%
3           39182      127      0.3%  <- muy desbalanceado
4           32570     6739     17.1%
5           39125      184      0.5%  <- muy desbalanceado
6           37377     1932      4.9%


## § 5b · Carga de datos adicionales

Escanea automáticamente todos los grupos dentro de `additional_data/` y los añade a `train_df`.

**Para añadir nuevos datos:** copia el grupo (carpeta con subcarpetas de secuencias + `Test conditions.xlsx`) dentro de `additional_data/` y re-ejecuta esta celda. No hace falta tocar nada más.

**Validación:** se ignoran secuencias con CSVs vacíos o solo cabecera para evitar errores en `read_all_test_data_from_path`.

In [6]:
def _validate_sequence(seq_path):
    """Devuelve True si el subdirectorio tiene al menos un CSV con datos reales."""
    csvs = [f for f in os.listdir(seq_path) if f.endswith('.csv')]
    if not csvs:
        return False
    for csv_file in csvs:
        try:
            lines = [l for l in open(os.path.join(seq_path, csv_file)).readlines() if l.strip()]
            if len(lines) <= 1:
                return False
        except Exception:
            return False
    return True


def _load_group(group_path, group_name):
    """
    Carga un grupo de datos adicionales desde group_path.
    Crea un directorio temporal con solo las secuencias validas,
    copia el xlsx y llama a read_all_test_data_from_path.
    Devuelve un DataFrame o None si falla.
    """
    xlsx      = os.path.join(group_path, 'Test conditions.xlsx')
    xlsx_copy = os.path.join(group_path, 'Test conditions copy.xlsx')

    if not os.path.exists(xlsx) and os.path.exists(xlsx_copy):
        try:
            shutil.copy(xlsx_copy, xlsx)
            print(f'    [info] copiado "Test conditions copy.xlsx" -> "Test conditions.xlsx"')
        except Exception as e:
            print(f'    [warn] no se pudo copiar el xlsx: {e}')

    if not os.path.exists(xlsx):
        print(f'    [skip] no hay Test conditions.xlsx en {group_name}')
        return None

    # Directorio temporal — forzar limpieza previa en Windows
    tmp = os.path.join(ADDITIONAL_BASE, f'_tmp_{group_name}')
    try:
        if os.path.exists(tmp):
            shutil.rmtree(tmp)
    except Exception as e:
        print(f'    [warn] no se pudo limpiar tmp: {e}')
    os.makedirs(tmp, exist_ok=True)
    shutil.copy(xlsx, os.path.join(tmp, 'Test conditions.xlsx'))

    subdirs = [d for d in os.listdir(group_path)
               if os.path.isdir(os.path.join(group_path, d))]
    n_ok, n_skip = 0, 0
    for sd in subdirs:
        sp  = os.path.join(group_path, sd)
        dst = os.path.join(tmp, sd)
        if _validate_sequence(sp):
            # dirs_exist_ok=True evita FileExistsError si el tmp no se limpió del todo
            shutil.copytree(sp, dst, dirs_exist_ok=True)
            n_ok += 1
        else:
            n_skip += 1

    print(f'    {n_ok} secuencias validas  |  {n_skip} omitidas (vacias)')

    result = None
    if n_ok > 0:
        try:
            result = read_all_test_data_from_path(tmp + os.sep, pre_processing, is_plot=False)
        except Exception as e:
            print(f'    [error] carga fallida: {e}')

    try:
        shutil.rmtree(tmp)
    except Exception:
        pass
    return result


# ── Escaneo automatico de grupos ──────────────────────────────────────────
if not os.path.exists(ADDITIONAL_BASE):
    print(f'Carpeta additional_data/ no encontrada en {ADDITIONAL_BASE}')
    print('train_df no se modifica.')
else:
    grupos_disponibles = sorted([
        d for d in os.listdir(ADDITIONAL_BASE)
        if os.path.isdir(os.path.join(ADDITIONAL_BASE, d))
        and not d.startswith('_tmp')
    ])

    print(f'Grupos detectados en additional_data/: {len(grupos_disponibles)}')
    print()

    n_filas_antes = len(train_df)
    extra_dfs     = []

    for grupo in grupos_disponibles:
        group_path = os.path.join(ADDITIONAL_BASE, grupo)
        print(f'  [{grupo}]')
        df = _load_group(group_path, grupo)
        if df is not None:
            extra_dfs.append(df)
            print(f'    -> cargado: {df.shape[0]} filas, {df["test_condition"].nunique()} secuencias')
        else:
            print(f'    -> no cargado')
        print()

    if extra_dfs:
        train_df = pd.concat([train_df] + extra_dfs, ignore_index=True)
        n_nuevas = len(train_df) - n_filas_antes
        print(f'train_df actualizado: {n_filas_antes} -> {len(train_df)} filas  (+{n_nuevas})')
        print(f'Secuencias totales  : {train_df["test_condition"].nunique()}')
    else:
        print('Ningun grupo adicional cargado. train_df sin cambios.')

    print()
    print('=== Distribucion de etiquetas (train_df combinado) ===')
    print(f'{"Motor":<8} {"Normal":>8} {"Fallo":>8} {"% fallo":>9}')
    print('-' * 38)
    for mid in range(1, 7):
        col = f'data_motor_{mid}_label'
        if col in train_df.columns:
            c0  = (train_df[col] == 0).sum()
            c1  = (train_df[col] == 1).sum()
            pct = 100 * c1 / (c0 + c1) if (c0 + c1) > 0 else 0
            flag = '  <- muy desbalanceado' if pct < 1.0 else ''
            print(f'{mid:<8} {c0:>8} {c1:>8} {pct:>8.1f}%{flag}')

Grupos detectados en additional_data/: 3

  [additional_data_20240524_group_6]
    [warn] no se pudo limpiar tmp: [WinError 5] Acceso denegado: 'c:\\Users\\oscar\\Desktop\\TDS INDUSTRY\\additional_data\\_tmp_additional_data_20240524_group_6\\20240524_094877'
    13 secuencias validas  |  1 omitidas (vacias)
    -> cargado: 26086 filas, 13 secuencias

  [additional_training_data_group_1]
    [warn] no se pudo limpiar tmp: [WinError 5] Acceso denegado: 'c:\\Users\\oscar\\Desktop\\TDS INDUSTRY\\additional_data\\_tmp_additional_training_data_group_1\\20240529_122361'
    13 secuencias validas  |  0 omitidas (vacias)
    -> cargado: 27592 filas, 13 secuencias

  [additional_training_data_group_7]
    [warn] no se pudo limpiar tmp: [WinError 5] Acceso denegado: 'c:\\Users\\oscar\\Desktop\\TDS INDUSTRY\\additional_data\\_tmp_additional_training_data_group_7\\20240604_101954'
    10 secuencias validas  |  0 omitidas (vacias)
    -> cargado: 6420 filas, 10 secuencias

train_df actualizado: 3930

## § 6 · Función `train_motor()`

Para cada motor prueba los **tres algoritmos** sobre el mismo split de validación:

| Algoritmo | Grid | Configs |
|-----------|------|---------|
| **LogisticRegression** | C × threshold × closing | 112 |
| **XGBoost** | lr × n_est × depth × mcw × threshold × closing | 672 |
| **IsolationForest** | percentile × closing | 22 |

**Flujo:**
1. Split 80/20 por secuencia, estratificado por presencia de fallo.
2. Inyección de fallos en train (versión inyectada para LR/XGB; IsolationForest usa datos sin inyección).
3. Grid search de los tres algoritmos.
4. El ganador global se guarda en `models[motor_id]`.

In [7]:
def train_motor(motor_id):
    SEP  = '=' * 65
    LINE = '-' * 65
    print(f'\n{SEP}')
    print(f'  MOTOR {motor_id}')
    print(f'{SEP}')

    features  = [f'data_motor_{motor_id}_{f}' for f in MOTOR_FEATURES[motor_id]]
    label_col = f'data_motor_{motor_id}_label'

    missing = [f for f in features if f not in train_df.columns]
    if missing:
        print(f'SKIP — columnas faltantes: {missing}'); return
    if label_col not in train_df.columns:
        print('SKIP — sin columna de etiqueta'); return

    # ── 1. Split por secuencia ─────────────────────────────────────────────
    motor_df  = train_df.dropna(subset=[label_col]).copy()
    sequences = motor_df['test_condition'].unique()
    fail_seqs = [s for s in sequences
                 if (motor_df[motor_df['test_condition'] == s][label_col] == 1).any()]
    norm_seqs = [s for s in sequences if s not in fail_seqs]

    print(f'\n[Split] {len(sequences)} secuencias: {len(fail_seqs)} con fallo, {len(norm_seqs)} normales')

    def _split(seqs, ratio=0.2):
        if len(seqs) > 1:
            nv = max(1, int(ratio * len(seqs)))
            return train_test_split(seqs, test_size=nv, random_state=42)
        return seqs, []

    tr_f, va_f = _split(fail_seqs)
    tr_n, va_n = _split(norm_seqs)
    train_seqs, val_seqs = tr_f + tr_n, va_f + va_n
    print(f'  -> train: {len(train_seqs):3d} ({len(tr_f)}F + {len(tr_n)}N)')
    print(f'  -> val  : {len(val_seqs):3d} ({len(va_f)}F + {len(va_n)}N)')

    raw_train = motor_df[motor_df['test_condition'].isin(train_seqs)].copy()
    raw_val   = motor_df[motor_df['test_condition'].isin(val_seqs)].copy()

    # ── 2. Inyeccion de fallos (version para LR/XGB) ───────────────────────
    temp_col      = f'data_motor_{motor_id}_temperature'
    n_injected    = 0
    raw_train_inj = raw_train.copy()
    if temp_col in raw_train_inj.columns:
        for seq in train_seqs:
            sm = raw_train_inj['test_condition'] == seq
            if not (raw_train_inj.loc[sm, label_col] == 1).any():
                continue
            raw_train_inj.loc[sm, temp_col] = inject_failure(
                raw_train_inj.loc[sm, temp_col].values,
                raw_train_inj.loc[sm, label_col].values
            )
            n_injected += 1
            t  = raw_train_inj.loc[sm, temp_col]
            dc = f'data_motor_{motor_id}_temperature_diff'
            if dc in raw_train_inj.columns:
                raw_train_inj.loc[sm, dc] = t.diff(20).fillna(0).values
            for w, suf in [(5, '5'), (20, '20')]:
                for stat in ['mean', 'max', 'std']:
                    rc = f'data_motor_{motor_id}_temperature_roll_{stat}_{suf}'
                    if rc not in raw_train_inj.columns: continue
                    if stat == 'mean' : raw_train_inj.loc[sm, rc] = t.rolling(w, min_periods=1).mean().values
                    elif stat == 'max': raw_train_inj.loc[sm, rc] = t.rolling(w, min_periods=1).max().values
                    else              : raw_train_inj.loc[sm, rc] = t.rolling(w, min_periods=1).std().fillna(0).values
    print(f'\n[Inyeccion] {n_injected} secuencias modificadas (LR y XGB); IsoForest usa datos originales')

    X_train_raw = raw_train[features]
    X_train_inj = raw_train_inj[features]
    y_train     = raw_train[label_col]
    n0, n1      = (y_train == 0).sum(), (y_train == 1).sum()
    total       = n0 + n1
    print(f'[Balance]  0: {n0:6d} ({100*n0/total:.1f}%)   1: {n1:5d} ({100*n1/total:.1f}%)')

    X_val = raw_val[features]  if val_seqs else None
    y_val = raw_val[label_col] if val_seqs else None

    ratio = min(n0 / n1, 20.0) if n1 > 0 else 1.0
    sw    = np.where(y_train == 1, ratio, 1.0)
    print(f'[Pesos supervisados] {ratio:.1f}x en clase 1 (cap 20x)')

    best_score = -1.0
    best_info  = {}
    best_model = None

    def _search_thresholds(probs_or_scores, y_v):
        if y_v is None:
            return -1.0, 0.5, True
        bs, bt, bc = -1.0, 0.5, True
        for t in THRESHOLDS:
            base = (probs_or_scores >= t).astype(int)
            for use_cl in CLOSING_OPTS:
                pred = binary_closing(base, np.ones(5)).astype(int) if use_cl else base
                sc   = f1_score(y_v, pred, pos_label=1, zero_division=0)
                if sc > bs:
                    bs, bt, bc = sc, t, use_cl
        return bs, bt, bc

    # ══════════════════════════════════════════════════════════════════════
    #  A · LogisticRegression
    # ══════════════════════════════════════════════════════════════════════
    n_lr_cfg = len(GRID_LR['C']) * len(THRESHOLDS) * len(CLOSING_OPTS)
    print(f'\n{LINE}')
    print(f'  [A] LogisticRegression  ({len(GRID_LR["C"])} C x {len(THRESHOLDS)} thr x {len(CLOSING_OPTS)} closing = {n_lr_cfg} configs)')
    print(f'{LINE}')
    lr_best_score, lr_best_info, lr_best_model = -1.0, {}, None
    for c_val in GRID_LR['C']:
        mdl = Pipeline([('sc', StandardScaler()),
                        ('lr', LogisticRegression(C=c_val, random_state=42, max_iter=1000))])
        mdl.fit(X_train_inj, y_train, lr__sample_weight=sw)
        probs = mdl.predict_proba(X_val)[:, 1] if X_val is not None else None
        sc, t, cl = _search_thresholds(probs, y_val) if probs is not None else (-1.0, 0.5, True)
        print(f'  C={c_val:<6}  val F1={sc:.4f}  (thr={t}, closing={cl})')
        if sc > lr_best_score:
            lr_best_score = sc
            lr_best_info  = {'algo': 'LR', 'C': c_val, 'threshold': t, 'closing': cl}
            lr_best_model = mdl
    print(f'  => mejor LR: F1={lr_best_score:.4f}')
    if lr_best_score > best_score:
        best_score, best_info, best_model = lr_best_score, lr_best_info, lr_best_model

    # ══════════════════════════════════════════════════════════════════════
    #  B · XGBoost
    # ══════════════════════════════════════════════════════════════════════
    ks, vs     = zip(*GRID_XGB.items())
    xgb_combos = [dict(zip(ks, v)) for v in itertools.product(*vs)]
    n_xgb_cfg  = len(xgb_combos) * len(THRESHOLDS) * len(CLOSING_OPTS)
    print(f'\n{LINE}')
    print(f'  [B] XGBoost  ({len(xgb_combos)} combos x {len(THRESHOLDS)} thr x {len(CLOSING_OPTS)} closing = {n_xgb_cfg} configs)')
    print(f'{LINE}')
    xgb_best_score, xgb_best_info, xgb_best_model = -1.0, {}, None
    for i, params in enumerate(xgb_combos):
        mdl = xgb.XGBClassifier(random_state=42, eval_metric='logloss',
                                 verbosity=0, tree_method='hist', **params)
        mdl.fit(X_train_inj, y_train, sample_weight=sw)
        probs = mdl.predict_proba(X_val)[:, 1] if X_val is not None else None
        sc, t, cl = _search_thresholds(probs, y_val) if probs is not None else (-1.0, 0.5, True)
        if sc > xgb_best_score:
            xgb_best_score = sc
            xgb_best_info  = {'algo': 'XGB', **params, 'threshold': t, 'closing': cl}
            xgb_best_model = mdl
        if (i + 1) % 6 == 0 or (i + 1) == len(xgb_combos):
            print(f'  combo {i+1:2d}/{len(xgb_combos)}  mejor F1 hasta ahora: {xgb_best_score:.4f}')
    print(f'  => mejor XGB: F1={xgb_best_score:.4f}')
    if xgb_best_score > best_score:
        best_score, best_info, best_model = xgb_best_score, xgb_best_info, xgb_best_model

    # ══════════════════════════════════════════════════════════════════════
    #  C · IsolationForest
    # ══════════════════════════════════════════════════════════════════════
    n_iso_cfg = len(ISO_PERCENTILES) * len(CLOSING_OPTS)
    print(f'\n{LINE}')
    print(f'  [C] IsolationForest  ({len(ISO_PERCENTILES)} percentiles x {len(CLOSING_OPTS)} closing = {n_iso_cfg} configs)')
    print(f'{LINE}')
    X_normal = X_train_raw[y_train == 0]
    iso = IsolationForest(n_estimators=200, contamination='auto', random_state=42)
    iso.fit(X_normal)
    train_scores_iso = -iso.decision_function(X_train_raw)
    thr_iso_list     = [float(np.percentile(train_scores_iso, p)) for p in ISO_PERCENTILES]
    iso_best_score, iso_best_info = -1.0, {}
    if X_val is not None:
        val_scores_iso = -iso.decision_function(X_val)
        nv0, nv1 = (y_val == 0).sum(), (y_val == 1).sum()
        print(f'  Val: {nv0} normales, {nv1} fallos')
        if nv1 > 0:
            ns, fs = val_scores_iso[y_val == 0], val_scores_iso[y_val == 1]
            sep_ok = fs.mean() > ns.mean()
            print(f'  normales media={ns.mean():.4f}  fallos media={fs.mean():.4f}  {"SEPARABLE" if sep_ok else "SIN SENAL"}')
        print(f'  {"Pct":>4} {"Thr":>9} {"Closing":>9} {"F1":>8}')
        print('  ' + '-' * 36)
        for p, t in zip(ISO_PERCENTILES, thr_iso_list):
            base = (val_scores_iso >= t).astype(int)
            for use_cl in CLOSING_OPTS:
                pred = binary_closing(base, np.ones(5)).astype(int) if use_cl else base
                sc   = f1_score(y_val, pred, pos_label=1, zero_division=0)
                mark = '  <- best' if sc > iso_best_score else ''
                if sc > iso_best_score:
                    iso_best_score = sc
                    iso_best_info  = {'algo': 'IsoForest', 'percentile': p, 'threshold': t, 'closing': use_cl}
                print(f'  {p:>4} {t:>9.4f} {str(use_cl):>9} {sc:>8.4f}{mark}')
    else:
        iso_best_info = {'algo': 'IsoForest', 'percentile': ISO_PERCENTILES[8],
                         'threshold': thr_iso_list[8], 'closing': True}
    print(f'  => mejor IsoForest: F1={iso_best_score:.4f}')
    if iso_best_score > best_score:
        best_score, best_info, best_model = iso_best_score, iso_best_info, iso

    # ── Ganador ───────────────────────────────────────────────────────────
    print(f'\n{SEP}')
    print(f'  GANADOR Motor {motor_id}: {best_info.get("algo", "?")}  |  val F1 = {best_score:.4f}')
    ranking = sorted([
        (lr_best_score,  'LR'),
        (xgb_best_score, 'XGB'),
        (iso_best_score, 'IsoForest'),
    ], reverse=True)
    print('  Ranking: ' + '  >  '.join(f'{a}: {s:.4f}' for s, a in ranking))
    print(f'  Config ganadora: {best_info}')
    print(f'{SEP}')

    # Guardar modelo, config completa y métricas de decisión
    models[motor_id] = {
        'type'       : best_info.get('algo', '?'),
        'model'      : best_model,
        'threshold'  : best_info.get('threshold', 0.5),
        'use_closing': best_info.get('closing', True),
        'config'     : best_info,          # hiperparámetros completos del ganador
        'val_f1'     : best_score,
        'ranking'    : {a: s for s, a in ranking},
    }
    val_f1_per_motor[motor_id] = best_score if X_val is not None else None

    def _apply(m_info, X):
        algo = m_info['type']
        thr  = m_info['threshold']
        cl   = m_info['use_closing']
        m    = m_info['model']
        if algo == 'IsoForest':
            pred = (-m.decision_function(X) >= thr).astype(int)
        else:
            pred = (m.predict_proba(X)[:, 1] >= thr).astype(int)
        return binary_closing(pred, np.ones(5)).astype(int) if cl else pred

    X_tr_report = X_train_raw if best_info.get('algo') == 'IsoForest' else X_train_inj
    print('\n--- Training report ---')
    print(classification_report(y_train, _apply(models[motor_id], X_tr_report), zero_division=0))
    if X_val is not None:
        print('--- Validation report ---')
        print(classification_report(y_val, _apply(models[motor_id], X_val), zero_division=0))


print('train_motor() definida')

train_motor() definida


## § 7–12 · Entrenamiento motor a motor

Cada celda es independiente: puedes re-ejecutar solo la del motor que estés tuneando.

In [8]:
# § 7 · Motor 1
train_motor(1)


  MOTOR 1

[Split] 59 secuencias: 7 con fallo, 52 normales
  -> train:  48 (6F + 42N)
  -> val  :  11 (1F + 10N)

[Inyeccion] 6 secuencias modificadas (LR y XGB); IsoForest usa datos originales
[Balance]  0:  81698 (97.0%)   1:  2523 (3.0%)
[Pesos supervisados] 20.0x en clase 1 (cap 20x)

-----------------------------------------------------------------
  [A] LogisticRegression  (4 C x 14 thr x 2 closing = 112 configs)
-----------------------------------------------------------------
  C=0.01    val F1=0.3187  (thr=0.6, closing=True)
  C=0.1     val F1=0.3089  (thr=0.6, closing=True)
  C=1.0     val F1=0.2901  (thr=0.5, closing=False)
  C=10.0    val F1=0.2903  (thr=0.5, closing=False)
  => mejor LR: F1=0.3187

-----------------------------------------------------------------
  [B] XGBoost  (24 combos x 14 thr x 2 closing = 672 configs)
-----------------------------------------------------------------
  combo  6/24  mejor F1 hasta ahora: 0.7133
  combo 12/24  mejor F1 hasta ahora: 0.7

In [9]:
# § 8 · Motor 2
train_motor(2)


  MOTOR 2

[Split] 59 secuencias: 6 con fallo, 53 normales
  -> train:  48 (5F + 43N)
  -> val  :  11 (1F + 10N)

[Inyeccion] 5 secuencias modificadas (LR y XGB); IsoForest usa datos originales
[Balance]  0:  84092 (98.3%)   1:  1444 (1.7%)
[Pesos supervisados] 20.0x en clase 1 (cap 20x)

-----------------------------------------------------------------
  [A] LogisticRegression  (4 C x 14 thr x 2 closing = 112 configs)
-----------------------------------------------------------------
  C=0.01    val F1=0.6476  (thr=0.005, closing=True)
  C=0.1     val F1=0.5663  (thr=0.005, closing=True)
  C=1.0     val F1=0.5662  (thr=0.005, closing=True)
  C=10.0    val F1=0.5662  (thr=0.005, closing=True)
  => mejor LR: F1=0.6476

-----------------------------------------------------------------
  [B] XGBoost  (24 combos x 14 thr x 2 closing = 672 configs)
-----------------------------------------------------------------
  combo  6/24  mejor F1 hasta ahora: 0.6484
  combo 12/24  mejor F1 hasta ahor

In [10]:
# § 9 · Motor 3
train_motor(3)


  MOTOR 3

[Split] 59 secuencias: 5 con fallo, 54 normales
  -> train:  48 (4F + 44N)
  -> val  :  11 (1F + 10N)

[Inyeccion] 4 secuencias modificadas (LR y XGB); IsoForest usa datos originales
[Balance]  0:  87974 (99.2%)   1:   751 (0.8%)
[Pesos supervisados] 20.0x en clase 1 (cap 20x)

-----------------------------------------------------------------
  [A] LogisticRegression  (4 C x 14 thr x 2 closing = 112 configs)
-----------------------------------------------------------------
  C=0.01    val F1=0.1894  (thr=0.4, closing=True)
  C=0.1     val F1=0.1931  (thr=0.4, closing=True)
  C=1.0     val F1=0.1559  (thr=0.2, closing=True)
  C=10.0    val F1=0.1574  (thr=0.5, closing=True)
  => mejor LR: F1=0.1931

-----------------------------------------------------------------
  [B] XGBoost  (24 combos x 14 thr x 2 closing = 672 configs)
-----------------------------------------------------------------
  combo  6/24  mejor F1 hasta ahora: 0.0777
  combo 12/24  mejor F1 hasta ahora: 0.077

In [11]:
# § 10 · Motor 4
train_motor(4)


  MOTOR 4

[Split] 59 secuencias: 6 con fallo, 53 normales
  -> train:  48 (5F + 43N)
  -> val  :  11 (1F + 10N)

[Inyeccion] 5 secuencias modificadas (LR y XGB); IsoForest usa datos originales
[Balance]  0:  81021 (97.6%)   1:  2022 (2.4%)
[Pesos supervisados] 20.0x en clase 1 (cap 20x)

-----------------------------------------------------------------
  [A] LogisticRegression  (4 C x 14 thr x 2 closing = 112 configs)
-----------------------------------------------------------------
  C=0.01    val F1=0.6224  (thr=0.15, closing=True)
  C=0.1     val F1=0.6312  (thr=0.15, closing=True)
  C=1.0     val F1=0.6435  (thr=0.15, closing=True)
  C=10.0    val F1=0.6550  (thr=0.15, closing=True)
  => mejor LR: F1=0.6550

-----------------------------------------------------------------
  [B] XGBoost  (24 combos x 14 thr x 2 closing = 672 configs)
-----------------------------------------------------------------
  combo  6/24  mejor F1 hasta ahora: 0.5781
  combo 12/24  mejor F1 hasta ahora: 0

In [12]:
# § 11 · Motor 5
train_motor(5)


  MOTOR 5

[Split] 59 secuencias: 5 con fallo, 54 normales
  -> train:  48 (4F + 44N)
  -> val  :  11 (1F + 10N)

[Inyeccion] 4 secuencias modificadas (LR y XGB); IsoForest usa datos originales
[Balance]  0:  87130 (98.3%)   1:  1508 (1.7%)
[Pesos supervisados] 20.0x en clase 1 (cap 20x)

-----------------------------------------------------------------
  [A] LogisticRegression  (4 C x 14 thr x 2 closing = 112 configs)
-----------------------------------------------------------------
  C=0.01    val F1=0.0759  (thr=0.3, closing=False)
  C=0.1     val F1=0.0797  (thr=0.3, closing=True)
  C=1.0     val F1=0.0912  (thr=0.5, closing=False)
  C=10.0    val F1=0.0937  (thr=0.5, closing=False)
  => mejor LR: F1=0.0937

-----------------------------------------------------------------
  [B] XGBoost  (24 combos x 14 thr x 2 closing = 672 configs)
-----------------------------------------------------------------
  combo  6/24  mejor F1 hasta ahora: 0.5683
  combo 12/24  mejor F1 hasta ahora: 0.

In [13]:
# § 12 · Motor 6
train_motor(6)


  MOTOR 6

[Split] 59 secuencias: 10 con fallo, 49 normales
  -> train:  48 (8F + 40N)
  -> val  :  11 (2F + 9N)

[Inyeccion] 8 secuencias modificadas (LR y XGB); IsoForest usa datos originales
[Balance]  0:  78708 (97.0%)   1:  2474 (3.0%)
[Pesos supervisados] 20.0x en clase 1 (cap 20x)

-----------------------------------------------------------------
  [A] LogisticRegression  (4 C x 14 thr x 2 closing = 112 configs)
-----------------------------------------------------------------
  C=0.01    val F1=0.3481  (thr=0.8, closing=True)
  C=0.1     val F1=0.3703  (thr=0.8, closing=False)
  C=1.0     val F1=0.3873  (thr=0.8, closing=False)
  C=10.0    val F1=0.3900  (thr=0.8, closing=False)
  => mejor LR: F1=0.3900

-----------------------------------------------------------------
  [B] XGBoost  (24 combos x 14 thr x 2 closing = 672 configs)
-----------------------------------------------------------------
  combo  6/24  mejor F1 hasta ahora: 0.0516
  combo 12/24  mejor F1 hasta ahora: 0.

## § 13 · Resumen global de F1 y algoritmo ganador

In [14]:
SEP  = '=' * 65
LINE = '-' * 65

# Etiquetas de hiperparámetros por algoritmo (en el orden que se quiere mostrar)
PARAM_LABELS = {
    'LR'       : ['C'],
    'XGB'      : ['learning_rate', 'n_estimators', 'max_depth', 'min_child_weight'],
    'IsoForest': ['percentile'],
}

print(SEP)
print('  RESUMEN FINAL — MODELO ÓPTIMO POR MOTOR')
print(SEP)

scored = {}
for mid in range(1, 7):
    info = models.get(mid)
    if info is None:
        print(f'\n  Motor {mid}  —  no entrenado')
        continue

    algo   = info['type']
    f1     = info.get('val_f1', val_f1_per_motor.get(mid))
    cfg    = info.get('config', {})
    rank   = info.get('ranking', {})
    bar    = '#' * int((f1 or 0) * 30)

    print(f'\n  Motor {mid}  |  {algo}  |  F1 = {f1:.4f}  |  {bar}')
    print(f'  {LINE}')

    # Hiperparámetros del modelo ganador
    params_to_show = PARAM_LABELS.get(algo, [])
    for key in params_to_show:
        val = cfg.get(key, '—')
        print(f'    {key:<20} {val}')

    # Parámetros de decisión (threshold y closing, comunes a todos)
    print(f'    {"threshold":<20} {cfg.get("threshold", info["threshold"])}')
    print(f'    {"closing":<20} {cfg.get("closing", info["use_closing"])}')

    # Ranking de los tres algoritmos
    if rank:
        rank_str = '  >  '.join(f'{a}: {rank[a]:.4f}' for a in sorted(rank, key=rank.get, reverse=True))
        print(f'    {"ranking":<20} {rank_str}')

    if f1 is not None:
        scored[mid] = f1

print()
print(SEP)
if scored:
    gf1 = sum(scored.values()) / len(scored)
    print(f'  Global mean F1 ({len(scored)} motores): {gf1:.4f}')
print(SEP)

  RESUMEN FINAL — MODELO ÓPTIMO POR MOTOR

  Motor 1  |  XGB  |  F1 = 0.7344  |  ######################
  -----------------------------------------------------------------
    learning_rate        0.02
    n_estimators         150
    max_depth            4
    min_child_weight     10
    threshold            0.03
    closing              True
    ranking              XGB: 0.7344  >  IsoForest: 0.3239  >  LR: 0.3187

  Motor 2  |  IsoForest  |  F1 = 0.8230  |  ########################
  -----------------------------------------------------------------
    percentile           75
    threshold            -0.041223834535505655
    closing              False
    ranking              IsoForest: 0.8230  >  XGB: 0.6484  >  LR: 0.6476

  Motor 3  |  LR  |  F1 = 0.1931  |  #####
  -----------------------------------------------------------------
    C                    0.1
    threshold            0.4
    closing              True
    ranking              LR: 0.1931  >  XGB: 0.0777  >  IsoFor

## § 14 · Generación del CSV de submission

`EXCLUDE_MOTOR = N` → ese motor se fuerza a `-1` (truco Kaggle, no gasta submission real).  
`EXCLUDE_MOTOR = None` → CSV de submission final completo.

In [17]:
EXCLUDE_MOTOR = 2

sample_sub = pd.read_csv(SAMPLE_SUB_PATH)
final_sub  = sample_sub.copy()

def _predict_motor(model_info, X):
    m   = model_info['model']
    thr = model_info['threshold']
    cl  = model_info['use_closing']
    if model_info['type'] == 'IsoForest':
        pred = (-m.decision_function(X) >= thr).astype(int)
    else:
        pred = (m.predict_proba(X)[:, 1] >= thr).astype(int)
    return binary_closing(pred, np.ones(5)).astype(int) if cl else pred


for test_id in sample_sub['test_condition'].unique():
    sub_mask = sample_sub['test_condition'] == test_id
    mtd      = test_df[test_df['test_condition'] == test_id].sort_values('time')
    exp_len  = int(sub_mask.sum())

    if len(mtd) == 0:
        for mid in range(1, 7):
            final_sub.loc[sub_mask, f'data_motor_{mid}_label'] = 0
        continue

    for mid in range(1, 7):
        feats = [f'data_motor_{mid}_{f}' for f in MOTOR_FEATURES[mid]]
        if mid not in models or not all(f in mtd.columns for f in feats):
            final_sub.loc[sub_mask, f'data_motor_{mid}_label'] = 0
            continue
        preds = _predict_motor(models[mid], mtd[feats])
        if len(preds) != exp_len:
            print(f'  Length mismatch {test_id} motor {mid}: {len(preds)} vs {exp_len}')
            preds = (preds[:exp_len] if len(preds) > exp_len
                     else np.pad(preds, (0, exp_len - len(preds)), 'constant'))
        final_sub.loc[sub_mask, f'data_motor_{mid}_label'] = preds

for mid in range(1, 7):
    final_sub[f'data_motor_{mid}_label'] = final_sub[f'data_motor_{mid}_label'].astype(int)

if EXCLUDE_MOTOR is not None:
    print(f'Truco Kaggle: motor {EXCLUDE_MOTOR} -> -1')
    final_sub[f'data_motor_{EXCLUDE_MOTOR}_label'] = -1
    out = os.path.join(SUBMISSION_DIR, f'motor_excluded_{EXCLUDE_MOTOR}_submission.csv')
else:
    out = os.path.join(SUBMISSION_DIR, 'motor_submission.csv')

final_sub.to_csv(out, index=False)
print(f'Guardado: {out}')
print(f'Shape   : {final_sub.shape}')
print()
print(f'{"Motor":<8} {"Algoritmo":<12} {"0 (normal)":>12} {"1 (fallo)":>10} {"% fallo":>9}')
print('-' * 56)
for mid in range(1, 7):
    col  = f'data_motor_{mid}_label'
    algo = models.get(mid, {}).get('type', '?')
    c0   = (final_sub[col] == 0).sum()
    c1   = (final_sub[col] == 1).sum()
    pct  = 100 * c1 / (c0 + c1) if (c0 + c1) > 0 else 0
    print(f'{mid:<8} {algo:<12} {c0:>12} {c1:>10} {pct:>8.1f}%')

final_sub.head()

Truco Kaggle: motor 2 -> -1
Guardado: c:\Users\oscar\Desktop\TDS INDUSTRY\submissions\motor_excluded_2_submission.csv
Shape   : (14157, 8)

Motor    Algoritmo      0 (normal)  1 (fallo)   % fallo
--------------------------------------------------------
1        XGB                 13699        458      3.2%
2        IsoForest               0          0      0.0%
3        LR                  13267        890      6.3%
4        LR                   8967       5190     36.7%
5        XGB                 14007        150      1.1%
6        IsoForest           14111         46      0.3%


,idx,data_motor_1_label,data_motor_2_label,data_motor_3_label,data_motor_4_label,data_motor_5_label,data_motor_6_label,test_condition
0,0,0,-1,0,0,0,0,20240527_094865
1,1,0,-1,0,0,0,0,20240527_094865
2,2,1,-1,0,0,0,0,20240527_094865
3,3,0,-1,0,0,0,0,20240527_094865
4,4,0,-1,0,0,0,0,20240527_094865
